In [ ]:
# Bronze Ingest — Notebook de respaldo
# Ejecutar manualmente si Airflow falla
# Descarga users_2021 de ClickHouse playground -> MinIO bronze/

In [ ]:
import os
os.environ["STORAGE_URI"] = "http://minio:9000"
os.environ["AWS_ACCESS_KEY_ID"] = "admin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "password"
os.environ["BRONZE_YEAR"] = "2021"
os.environ["BRONZE_LIMIT"] = "50000" 

In [ ]:
# Cargar datos manuales (posts 2020/2021, users 2020)
import sys
sys.path.insert(0, "/home/jovyan/scripts")
from bronze_manual_load import load_posts, load_users

print("Cargando datos manuales...")
load_posts("2020")
load_posts("2021")
load_users("2020")
print("Carga manual completada.")

In [ ]:
# Ejecutar ingesta Bronze users_2021 (la del pipeline)
from bronze_ingest_users import run as bronze_run
bronze_run(year="2021", limit=50000)

In [ ]:
# Verificar archivos en MinIO
import boto3
s3 = boto3.client("s3", endpoint_url="http://minio:9000",
                   aws_access_key_id="admin", aws_secret_access_key="password",
                   region_name="us-east-1")
for prefix in ["users/2020/", "users/2021/", "posts/2020/", "posts/2021/"]:
    objs = s3.list_objects_v2(Bucket="bronze", Prefix=prefix)
    keys = [o["Key"] for o in objs.get("Contents", [])]
    print(f"bronze/{prefix}: {keys}")